<h1 style="font-size:24px;">Prediction</h1>

In [ ]:
import numpy as np
from scipy import stats
from sklearn.model_selection import RepeatedKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from cca_zoo.linear import rCCA


def grid_search_rcca_best_c(X, Y, param_c, cvg):
    n_samples = X.shape[0]
    X_scores = np.zeros((n_samples, len(param_c)))
    Y_scores = np.zeros((n_samples, len(param_c)))

    for tr_idx, ts_idx in cvg.split(X):
        tr_X, tr_Y = X[tr_idx], Y[tr_idx]
        ts_X, ts_Y = X[ts_idx], Y[ts_idx]

        for ic, c in enumerate(param_c):
            model = rCCA(latent_dimensions=1, c=c)
            model.fit((tr_X, tr_Y))
            a, b = model.transform((ts_X, ts_Y))

            X_scores[ts_idx, ic] = a[:, 0]
            Y_scores[ts_idx, ic] = b[:, 0]

    rvals = np.zeros(len(param_c))

    for ic in range(len(param_c)):
        r, _ = stats.pearsonr(X_scores[:, ic], Y_scores[:, ic])
        rvals[ic] = r

    sidx = np.where(rvals == np.nanmax(rvals))[0]
    return np.median(param_c[sidx])


def cross_validation_cca(X, Y, cvg, comp_num=1, site_labels=None):

    n_subjects = X.shape[0]

    avec = np.zeros((n_subjects, comp_num))
    bvec = np.zeros((n_subjects, comp_num))

    for tr_idx, ts_idx in cvg.split(X, site_labels):

        tr_X, tr_Y = X[tr_idx], Y[tr_idx]
        ts_X, ts_Y = X[ts_idx], Y[ts_idx]

        Y_scaler = StandardScaler()
        tr_Y = Y_scaler.fit_transform(tr_Y)
        ts_Y = Y_scaler.transform(ts_Y)

        inner_cvg = RepeatedKFold(n_splits=10, n_repeats=1, random_state=1)

        best_c = grid_search_rcca_best_c(
            tr_X, tr_Y, np.arange(0.0, 1.0, 0.05), inner_cvg
        )

        model = rCCA(latent_dimensions=comp_num, c=best_c)
        model.fit((tr_X, tr_Y))
        a, b = model.transform((ts_X, ts_Y))
        avec[ts_idx] = a
        bvec[ts_idx] = b

    return avec, bvec


# X: predictor matrix (n_subjects × n_features)
# Y: response matrix (n_subjects × n_features)

X = your_predictor_matrix
Y = your_response_matrix

rep_num = 5
comp_num = 1

X = stats.zscore(X, axis=0)
site_labels = np.zeros((X.shape[0],))
n_subjects = X.shape[0]

print(X.shape)
print(Y.shape)

all_avec = np.zeros((rep_num, n_subjects, comp_num))
all_bvec = np.zeros((rep_num, n_subjects, comp_num))

for irep in range(rep_num):
    print(irep)
    cvg = StratifiedKFold(
        n_splits=10,
        random_state=irep + 1000,
        shuffle=True
    )

    avec, bvec = cross_validation_cca(
        X, Y,
        cvg,
        comp_num=comp_num,
        site_labels=site_labels
    )

    all_avec[irep] = avec
    all_bvec[irep] = bvec


real_rvals = np.nan * np.zeros((rep_num, comp_num))
for irep in range(rep_num):
    for i in range(comp_num):
        r, p = stats.pearsonr(
            all_avec[irep, :, i],
            all_bvec[irep, :, i]
        )
        print(f"rep {irep}, comp {i}: r = {r:.4f}, p = {p:.4g}")
        real_rvals[irep, i] = r

print("Median r across repetitions:")
print(np.nanmedian(real_rvals, axis=0))
real_rvals

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import t

# Color palette
palette = sns.color_palette("deep")
dot_color = palette[1]

# Aggregate across repetitions
all_true_labels = np.median(all_avec, axis=0).flatten()
all_pred_labels = np.median(all_bvec, axis=0).flatten()

plt.figure(figsize=(8, 6))

# Scatter plot
plt.scatter(
    all_true_labels,
    all_pred_labels,
    color=dot_color,
    alpha=0.65,
    edgecolor="k",
    linewidth=0.4
)

# Fit regression line
z = np.polyfit(all_true_labels, all_pred_labels, 1)
p = np.poly1d(z)
y_fit = p(all_true_labels)

# Compute 95% confidence interval
n = len(all_true_labels)
sigma = np.std(all_pred_labels - y_fit)
x_mean = np.mean(all_true_labels)
x_ss = np.sum((all_true_labels - x_mean) ** 2)

confidence = 0.95
alpha = 1 - confidence
t_val = t.ppf(1 - alpha / 2, n - 2)

ci_upper = []
ci_lower = []

for xi in all_true_labels:
    se = sigma * np.sqrt(1 / n + ((xi - x_mean) ** 2) / x_ss)
    ci_upper.append(p(xi) + t_val * se)
    ci_lower.append(p(xi) - t_val * se)

ci_upper = np.array(ci_upper)
ci_lower = np.array(ci_lower)

# Sort values for plotting
idx = np.argsort(all_true_labels)
sorted_x = all_true_labels[idx]
sorted_y_fit = y_fit[idx]
sorted_ci_upper = ci_upper[idx]
sorted_ci_lower = ci_lower[idx]

# Regression line and confidence band
plt.plot(sorted_x, sorted_y_fit, color="black", linewidth=2)
plt.fill_between(
    sorted_x,
    sorted_ci_lower,
    sorted_ci_upper,
    color="gray",
    alpha=0.2
)

# Figure style
plt.xlabel("True Values", fontsize=14)
plt.ylabel("Predicted Values", fontsize=14)
plt.grid(False)

ax = plt.gca()
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)

plt.tight_layout()
# plt.savefig("prediction_scatter.svg", format="svg", dpi=300, bbox_inches="tight")
plt.show()